# Question 1: Which genres share audiences?

*What does the overlap reveal about relationships between musical genres?*

**Step 1: build the user x genre table** from the 600-user test sample (Lecture 4, preprocessing).

Run the cells from top to bottom. The notebook expects to be run from the `Hans` folder.

## A. Load the data

One row in the CSV is one listen. We only need who listened (`user_id`) and the genre label.

In [12]:
from pathlib import Path
import pandas as pd

# Path is built from the folder this notebook sits in (Hans), so it works on any computer.
CSV = Path.cwd().parent / "Samples" / "01_Test_Sample (600 users)" / "Output" / "sample_2023_600users.csv"

listens = pd.read_csv(CSV, usecols=["user_id", "genre"])
print("rows (listens):", len(listens))
print("users:", listens["user_id"].nunique())
print("share of listens WITH a genre:", round(listens["genre"].notna().mean(), 3))

rows (listens): 200007
users: 600
share of listens WITH a genre: 0.611


Listens without a genre cannot tell us anything about genres, so we drop them.
This means the table describes only the ~61% of listening that has a genre.

In [13]:
listens = listens.dropna(subset=["genre"])
listens.head()

,user_id,genre
0,13,downtempo
1,13,modern classical
3,13,electronic
4,13,disco
5,13,synth-pop


## B. Count listens per (user, genre)

`groupby(...).size()` gives a long list with one number per user-genre pair that occurs.
`unstack()` then turns the genre level into columns, so rows = users and columns = genres.
Pairs that never occur become `NaN`, and `fill_value=0` says "this user played 0 listens of that genre".

In [14]:
counts_long = listens.groupby(["user_id", "genre"]).size()
counts_long.head()

user_id  genre            
13       acid house           1
         afro house           1
         aggrotech            2
         alternative metal    1
         alternative rock     5
dtype: int64

In [11]:
counts = counts_long.unstack(fill_value=0)
print("table shape (users x genres):", counts.shape)
counts.iloc[:5, :6]

table shape (users x genres): (600, 830)


genre,2 tone,2-step,abstract hip hop,acid house,acid jazz,acid rock
user_id,,,,,,
13,0,0,0,1,0,0
59,0,0,0,0,0,0
106,0,0,0,0,0,0
173,0,0,0,0,0,0
237,0,0,0,0,0,0


## C. Turn counts into shares

`counts.sum(axis=1)` is each user's total genre-labelled listens.
`div(..., axis=0)` divides every row by its own total (`axis=0` = match on the row index).
After this, each user's row sums to 1.

In [6]:
shares = counts.div(counts.sum(axis=1), axis=0)
print("share of cells that are zero:", round((shares.values == 0).mean(), 3))
shares.sum(axis=1).head()

share of cells that are zero: 0.946


user_id
13     1.0
59     1.0
106    1.0
173    1.0
237    1.0
dtype: float64

In [ ]:
# D. Genre overview: which genres are big, and which are tiny?
overview = pd.DataFrame({
    "listens": counts.sum(),                 # total listens in this genre
    "users_any": (counts > 0).sum(),         # users with at least 1 listen in it
    "users_2pct": (shares >= 0.02).sum(),    # users who really play it (>= 2% of their listens)
})
overview["share_of_listens"] = overview["listens"] / overview["listens"].sum()
overview = overview.sort_values("listens", ascending=False)
print(overview.head(15).round(3))

# How concentrated is it? Cumulative share of listens by genre rank.
cum = overview["share_of_listens"].cumsum()
for p in (0.5, 0.8, 0.95):
    print(f"{p:.0%} of listens are in the top {(cum < p).sum() + 1} genres (of {len(overview)})")

# Cutoff for which genres to keep. Change MIN_USERS_SHARE and see what happens.
MIN_USERS_SHARE = 0.07   # keep a genre if >= 7% of users really play it
keep = overview[overview["users_2pct"] / len(shares) >= MIN_USERS_SHARE].index
print(f"\nKeeping {len(keep)} genres; they hold {overview.loc[keep, 'share_of_listens'].sum():.0%} of all listens")

### Try this
1. Change `axis=0` to `axis=1` in the cell above and run it again. What happens, and why?
2. Look at one user's row: `shares.loc[173].sort_values(ascending=False).head(10)`. Do the top genres sum to 1?